# Least privilege and access control models

# Coding problem, 4.2.d: Access-request auditor

**Skill:** Mitigate Risk · **Maps to:** 4.2.B, 4.2.C · **Language:** Python 3 (pytest)

## Problem
A security team reviews access requests before granting them. Each request either has a business
justification attached or it does not; the requester's access may or may not have been reviewed
recently; and the requested access may be scoped to a temporary time window or be standing
(permanent). Build the auditor function that applies the least-privilege triage rules to a single
request and returns the decision.

## Function contract
Implement `def solve(request: dict) -> str` in the code editor (see the Reference section below).

## Input
A dict with keys `current_role` (str), `requested_access` (str), `justification_present` (bool),
`time_bound` (bool or None), `last_reviewed_days_ago` (int). Optional keys default as documented
in the Reference section below (missing `justification_present` defaults to `False`; missing `time_bound`
defaults to `False`; missing `last_reviewed_days_ago` defaults to `0`).

## Output
One of the strings: `"approve"`, `"deny_no_justification"`, `"flag_privilege_creep"`,
`"approve_time_bound"`.

## Decision rule (apply in this exact order, first match wins)
1. No justification (`justification_present` is `False`) → `"deny_no_justification"`.
2. Else, stale review (`last_reviewed_days_ago > 365`) → `"flag_privilege_creep"`.
3. Else, time-bound request (`time_bound` is `True`) → `"approve_time_bound"`.
4. Else → `"approve"`.

Rule 1 is checked before rule 2, a request with no justification is denied even if it is also
years overdue for review or time-bound. The staleness threshold is 365 days, strictly greater
than (365 itself is still considered current).

## Examples
```
{"current_role": "Marketing Intern", "requested_access": "prod-db-write",
 "justification_present": False, "time_bound": False, "last_reviewed_days_ago": 10}
    -> "deny_no_justification"

{"current_role": "HR Analyst", "requested_access": "payroll-admin",
 "justification_present": True, "time_bound": False, "last_reviewed_days_ago": 800}
    -> "flag_privilege_creep"

{"current_role": "On-call Engineer", "requested_access": "prod-ssh",
 "justification_present": True, "time_bound": True, "last_reviewed_days_ago": 5}
    -> "approve_time_bound"

{"current_role": "Analyst", "requested_access": "reporting-read",
 "justification_present": True, "time_bound": False, "last_reviewed_days_ago": 30}
    -> "approve"
```

## Constraints
- Deterministic, offline, standard library only.
- Missing optional keys must not raise, use the documented defaults.
- Exactly one label returned (string, not list).
- Must pass all cases in the tests (visible + hidden).

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def solve(request: dict) -> str
```

### Input (`request`)
A dict describing one access request:

| key | type | required | default if missing | meaning |
|---|---|---|---|---|
| `current_role` | str | no | `""` | the requester's current job role (informational; not used in the decision below, but always present in well-formed data) |
| `requested_access` | str | no | `""` | the resource/access level being requested (informational) |
| `justification_present` | bool | yes | `False` | whether a business justification was submitted with the request |
| `time_bound` | bool or None | no | `False` | whether the requested access is scoped to a fixed, temporary time window; `None` is treated the same as `False` |
| `last_reviewed_days_ago` | int | no | `0` | days since this requester's access was last reviewed/recertified |

Missing optional keys default as shown above. `justification_present` is required conceptually
(it drives the first rule) but if the key is absent, treat it as `False` (fail closed).

### Output (`result`)
Exactly one string from `{"approve", "deny_no_justification", "flag_privilege_creep",
"approve_time_bound"}`.

### Decision rule (exact priority order, evaluate top to bottom, return on first match)
1. If `justification_present` is `False` (or missing) → return `"deny_no_justification"`.
   This is checked FIRST and overrides every other condition, including privilege creep.
2. Else if `last_reviewed_days_ago > 365` → return `"flag_privilege_creep"`.
   A stale access review (more than 365 days since last review) takes priority over approving,
   even if the request is time-bound. The threshold is a strict greater-than: exactly `365` does
   NOT trigger this rule.
3. Else if `time_bound` is `True` → return `"approve_time_bound"`.
4. Else → return `"approve"`.

### Guarantees / rules
- Pure function, deterministic, no I/O, no network.
- Standard library only unless the hidden grading tests: allow_deps` lists otherwise.
- Missing optional keys use the defaults in the table above; do not raise on missing keys.
- `last_reviewed_days_ago == 365` is NOT stale (rule 2 requires strictly greater than 365).

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Your implementation
"""Starter code for 4.2.d. Student completes the TODO.
This file intentionally FAILS the tests until implemented."""
from typing import Any


def solve(request: dict) -> str:
    """Audit a single access request per the least-privilege priority rules.

    Args:
        request: dict with keys current_role (str), requested_access (str),
            justification_present (bool), time_bound (bool or None),
            last_reviewed_days_ago (int). Optional keys default per the Reference section below.

    Returns:
        One of: "approve", "deny_no_justification", "flag_privilege_creep",
        "approve_time_bound".
    """
    # TODO: implement per the problem statement above
    raise NotImplementedError("implement solve() — see the problem statement above")


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

# ---- visible cases -------------------------------------------------------
def test_example_1_marketing_intern_no_justification():
    req = {"current_role": "Marketing Intern", "requested_access": "prod-db-write",
           "justification_present": False, "time_bound": False,
           "last_reviewed_days_ago": 10}
    assert impl.solve(req) == "deny_no_justification"


def test_example_2_hr_privilege_creep():
    req = {"current_role": "HR Analyst", "requested_access": "payroll-admin",
           "justification_present": True, "time_bound": False,
           "last_reviewed_days_ago": 800}
    assert impl.solve(req) == "flag_privilege_creep"


def test_example_3_oncall_time_bound():
    req = {"current_role": "On-call Engineer", "requested_access": "prod-ssh",
           "justification_present": True, "time_bound": True,
           "last_reviewed_days_ago": 5}
    assert impl.solve(req) == "approve_time_bound"


def test_example_4_plain_approve():
    req = {"current_role": "Analyst", "requested_access": "reporting-read",
           "justification_present": True, "time_bound": False,
           "last_reviewed_days_ago": 30}
    assert impl.solve(req) == "approve"


# ---- hidden / edge cases -------------------------------------------------
def test_boundary_exactly_365_is_not_stale():
    req = {"current_role": "Analyst", "requested_access": "reporting-read",
           "justification_present": True, "time_bound": False,
           "last_reviewed_days_ago": 365}
    assert impl.solve(req) == "approve"


def test_boundary_366_is_stale():
    req = {"current_role": "Analyst", "requested_access": "reporting-read",
           "justification_present": True, "time_bound": False,
           "last_reviewed_days_ago": 366}
    assert impl.solve(req) == "flag_privilege_creep"


def test_priority_no_justification_beats_privilege_creep():
    # justification_present False AND last_reviewed_days_ago > 365:
    # rule 1 is checked first, so deny_no_justification wins.
    req = {"current_role": "HR Analyst", "requested_access": "payroll-admin",
           "justification_present": False, "time_bound": False,
           "last_reviewed_days_ago": 900}
    assert impl.solve(req) == "deny_no_justification"


def test_priority_privilege_creep_beats_time_bound():
    req = {"current_role": "Contractor", "requested_access": "prod-ssh",
           "justification_present": True, "time_bound": True,
           "last_reviewed_days_ago": 400}
    assert impl.solve(req) == "flag_privilege_creep"


def test_missing_justification_key_defaults_false():
    req = {"current_role": "Intern", "requested_access": "prod-db-write",
           "time_bound": False, "last_reviewed_days_ago": 10}
    assert impl.solve(req) == "deny_no_justification"


def test_missing_time_bound_and_review_keys_default_to_approve():
    req = {"current_role": "Analyst", "requested_access": "reporting-read",
           "justification_present": True}
    assert impl.solve(req) == "approve"


def test_time_bound_none_treated_as_false():
    req = {"current_role": "Analyst", "requested_access": "reporting-read",
           "justification_present": True, "time_bound": None,
           "last_reviewed_days_ago": 5}
    assert impl.solve(req) == "approve"


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
